<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 4: Ab Testi

**VERİ BİLİMİ TEMELLERİ** · Modül 4 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta04/hafta04_ab_testi.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta04/hafta04_ab_testi.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>

</div>

# Hafta 4 — A/B Testi: Mobil Oyun Reklam Stratejisi

## Senaryo

Bir mobil oyun şirketi, oyun içi reklam stratejisini optimize etmek istiyor. İki farklı reklam modeli test ediliyor:

- **Grup A — Geçilebilir Reklam:** Kullanıcı 5 saniye sonra reklamı geçebilir.
- **Grup B — Ödüllü Reklam:** Kullanıcı reklamı izlerse oyun içi ödül kazanır.

**Araştırma Sorusu:** Hangi reklam modeli kullanıcıları oyunda daha uzun tutar?

**Bağımlı Değişken:** Ortalama oturum süresi (dakika)

---

## 1. Kütüphanelerin Yüklenmesi

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `scipy` | Bilimsel hesaplama ve istatistik testleri |
| `seaborn` | İstatistiksel görselleştirme (Matplotlib üzerine kurulu) |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
plt.rcParams['axes.unicode_minus'] = False

## 2. Sentetik Verinin Oluşturulması

Her iki grup için 500'er kullanıcı verisi oluşturuyoruz.

- **Grup A (Geçilebilir):** Ortalama 25 dk, standart sapma 8 dk
- **Grup B (Ödüllü):** Ortalama 28 dk, standart sapma 7 dk

In [ ]:
np.random.seed(42)

# Grup A: Geçilebilir Reklam
grup_a = np.random.normal(25, 8, 500)

# Grup B: Ödüllü Reklam
grup_b = np.random.normal(28, 7, 500)

# Negatif süreleri sıfıra çekiyoruz (oturum süresi negatif olamaz)
grup_a = np.maximum(grup_a, 0)
grup_b = np.maximum(grup_b, 0)

# DataFrame oluşturma
df = pd.DataFrame({
    'oturum_suresi': np.concatenate([grup_a, grup_b]),
    'grup': ['Geçilebilir Reklam'] * 500 + ['Ödüllü Reklam'] * 500
})

print(f"Toplam veri sayısı: {len(df)}")
df.head()

## 3. Keşifsel Veri Analizi

### Veri Gruplama ve Analiz

Verileri belirli kategorilere göre grupladıp özet istatistikler hesaplıyoruz. `groupby()` ile SQL'deki GROUP BY benzeri operasyonlar yapıyoruz.

In [ ]:
# Grup bazlı betimsel istatistik
ozet = df.groupby('grup')['oturum_suresi'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
ozet.columns = ['Gözlem Sayısı', 'Ortalama', 'Medyan', 'Standart Sapma', 'Minimum', 'Maksimum']
ozet.round(2)

In [ ]:
print(f"Ortalama Fark (B - A): {grup_b.mean() - grup_a.mean():.2f} dakika")

## 4. Görselleştirme

### 4.1 Histogramlar

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(grup_a, bins=30, alpha=0.6, color='#e74c3c', edgecolor='white', label='Geçilebilir Reklam (A)')
ax.hist(grup_b, bins=30, alpha=0.6, color='#2ecc71', edgecolor='white', label='Ödüllü Reklam (B)')

ax.axvline(grup_a.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'A Ortalama: {grup_a.mean():.1f} dk')
ax.axvline(grup_b.mean(), color='#2ecc71', linestyle='--', linewidth=2, label=f'B Ortalama: {grup_b.mean():.1f} dk')

ax.set_title('Oturum Süresi Dağılımı — Grup Karşılaştırması')
ax.set_xlabel('Oturum Süresi (dakika)')
ax.set_ylabel('Frekans')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 4.2 Kutu Grafiği (Boxplot)

### Kutu Grafiği (Box Plot)

Kutu grafiği verinin dağılımını, medyanını, çeyrekliklerini ve uç değerlerini (outlier) gösterir. Gruplar arası karşılaştırma için idealdir.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sns.boxplot(data=df, x='grup', y='oturum_suresi', palette=['#e74c3c', '#2ecc71'], ax=ax)

ax.set_title('Oturum Süresi — Kutu Grafiği')
ax.set_xlabel('Reklam Grubu')
ax.set_ylabel('Oturum Süresi (dakika)')
plt.tight_layout()
plt.show()

### 4.3 Keman Grafiği (Violin Plot)

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sns.violinplot(data=df, x='grup', y='oturum_suresi', palette=['#e74c3c', '#2ecc71'],
               inner='quartile', ax=ax)

ax.set_title('Oturum Süresi — Keman Grafiği')
ax.set_xlabel('Reklam Grubu')
ax.set_ylabel('Oturum Süresi (dakika)')
plt.tight_layout()
plt.show()

## 5. İstatistiksel Test

### 5.1 Normallik Kontrolü (Shapiro-Wilk)

Bağımsız örneklem T-testi uygulamadan önce, her iki grubun normal dağılıma uygunluğunu kontrol etmeliyiz.

In [ ]:
# Not: Shapiro-Wilk testi büyük örneklemlerde çok hassas olabilir.
# Bu yüzden ilk 100 gözlemi kullanıyoruz (veya tamamını da kullanabiliriz).

stat_a, p_a = stats.shapiro(grup_a)
stat_b, p_b = stats.shapiro(grup_b)

print("NORMALLİK TESTİ — Shapiro-Wilk")
print("=" * 50)
print(f"Grup A (Geçilebilir): W={stat_a:.4f}, p={p_a:.4f} → {'Normal' if p_a >= 0.05 else 'Normal değil'}")
print(f"Grup B (Ödüllü):      W={stat_b:.4f}, p={p_b:.4f} → {'Normal' if p_b >= 0.05 else 'Normal değil'}")
print()
if p_a >= 0.05 and p_b >= 0.05:
    print("Her iki grup da normal dağılıma uygun. T-testi uygulanabilir.")
else:
    print("Not: Büyük örneklemlerde Shapiro-Wilk çok hassas olabilir.")
    print("Merkezi Limit Teoremi'ne göre (n > 30) T-testi yine de uygulanabilir.")

### 5.2 Bağımsız Örneklem T-Testi

**Hipotezler:**
- H₀: İki grubun oturum süresi ortalamaları arasında fark yoktur (μA = μB)
- H₁: İki grubun oturum süresi ortalamaları arasında fark vardır (μA ≠ μB)
- Anlamlılık düzeyi: α = 0.05

In [ ]:
t_stat, p_value = stats.ttest_ind(grup_a, grup_b)

print("BAĞIMSIZ ÖRNEKLEM T-TESTİ")
print("=" * 50)
print(f"t İstatistiği: {t_stat:.4f}")
print(f"p-değeri:      {p_value:.6f}")
print(f"α (alfa):      0.05")
print()

if p_value < 0.05:
    print("KARAR: p < 0.05 → H₀ reddedilir.")
    print("İki grup arasında istatistiksel olarak ANLAMLI bir fark vardır.")
else:
    print("KARAR: p >= 0.05 → H₀ reddedilemez.")
    print("İki grup arasında istatistiksel olarak anlamlı bir fark yoktur.")

### 5.3 Etki Büyüklüğü — Cohen's d

p-değeri sadece farkın var olup olmadığını söyler. **Etki büyüklüğü** ise farkın ne kadar büyük olduğunu gösterir.

| Cohen's d | Yorum |
|---|---|
| 0.2 | Küçük etki |
| 0.5 | Orta etki |
| 0.8 | Büyük etki |

In [ ]:
def cohens_d(grup1, grup2):
    """Cohen's d etki büyüklüğünü hesaplar."""
    n1, n2 = len(grup1), len(grup2)
    var1, var2 = np.var(grup1, ddof=1), np.var(grup2, ddof=1)
    # Havuzlanmış standart sapma
    sp = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (np.mean(grup2) - np.mean(grup1)) / sp

d = cohens_d(grup_a, grup_b)

print("ETKİ BÜYÜKLÜĞÜ — Cohen's d")
print("=" * 50)
print(f"Cohen's d: {d:.4f}")

if abs(d) < 0.2:
    yorum = "Çok küçük etki"
elif abs(d) < 0.5:
    yorum = "Küçük etki"
elif abs(d) < 0.8:
    yorum = "Orta etki"
else:
    yorum = "Büyük etki"

print(f"Yorum:     {yorum}")
print(f"\nÖdüllü reklam grubu, geçilebilir reklam grubuna göre")
print(f"ortalama {grup_b.mean() - grup_a.mean():.1f} dakika daha uzun oturum süresine sahiptir.")

## 6. Sonuçların Görselleştirilmesi

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ortalamalar = [grup_a.mean(), grup_b.mean()]
hatalar = [stats.sem(grup_a) * 1.96, stats.sem(grup_b) * 1.96]  # %95 güven aralığı
renkler = ['#e74c3c', '#2ecc71']
etiketler = ['Geçilebilir Reklam\n(Grup A)', 'Ödüllü Reklam\n(Grup B)']

bars = ax.bar(etiketler, ortalamalar, yerr=hatalar, capsize=8,
              color=renkler, edgecolor='white', linewidth=2, alpha=0.85)

for bar, ort in zip(bars, ortalamalar):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{ort:.1f} dk', ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.set_title(f'Ortalama Oturum Süresi Karşılaştırması\n(p = {p_value:.4f}, d = {d:.2f})')
ax.set_ylabel('Ortalama Oturum Süresi (dakika)')
ax.set_ylim(0, max(ortalamalar) * 1.3)
plt.tight_layout()
plt.show()

## 7. Sonuç ve Yorum

### A/B Testi Sonuçları

| Metrik | Değer |
|---|---|
| Geçilebilir Reklam (A) Ortalama | ~25 dakika |
| Ödüllü Reklam (B) Ortalama | ~28 dakika |
| t İstatistiği | İstatistiksel olarak anlamlı |
| p-değeri | < 0.05 |
| Cohen's d | Küçük-Orta etki |

### Yorumlar

1. **İstatistiksel Anlamlılık:** Bağımsız örneklem T-testi sonucunda p < 0.05 bulunmuştur. Bu, iki grup arasındaki farkın tesadüfi olmadığını gösterir.

2. **Pratik Anlamlılık:** Cohen's d değeri küçük-orta düzeyde bir etki büyüklüğüne işaret eder. Ödüllü reklam modeli kullanıcıları yaklaşık 3 dakika daha uzun tutmaktadır.

3. **İş Kararı:** Ödüllü reklam modeli, kullanıcı bağlılığını artırmakta daha etkilidir. Ancak gelir etkisi, kullanıcı memnuniyeti ve uzun vadeli retansiyon da değerlendirilmelidir.

### Öneriler
- Ödüllü reklam modeline geçiş düşünülmelidir.
- Daha uzun süreli bir test (2-4 hafta) ile sonuçlar doğrulanmalıdır.
- Gelir ve kullanıcı kaybı metrikleri de analiz edilmelidir.

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

© 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>